# Category Consolidation

This notebook aims to consolidate our target variable, categories. The goal is to create a target variable with fewer unique categories by grouping them together into broader categories. We hope that doing so will improve our predictions as well as speed up our models.

In [1]:
import pandas as pd
import gc

data = pd.read_csv('../data/data_unencoded.csv')

data[['order_month', 'order_day', 'order_year']] = data['Order Date'].str.split('/', expand=True)
data = data.drop('Order Date', axis=1)
data[['order_month', 'order_day', 'order_year']] = data[['order_month', 'order_day', 'order_year']].apply(pd.to_numeric)

cols_to_drop = [
    'Survey ResponseID',
    'sell-YOUR-data',
    'sell-consumer-data',
    'small-biz-use',
    'census-use',
    'research-society'
]
data = data.drop(columns=cols_to_drop)
print(f'Loaded: {data.shape} | Unique categories: {data["Category"].nunique()}')

Loaded: (157026, 26) | Unique categories: 1625


In [2]:
# Build one text descriptor per category using up to 10 unique product titles.
# This groupby is the only step that touches all 157K rows — everything after works on 1,625 rows.
descriptors = (
    data.groupby('Category')['Title']
    .apply(lambda x: '; '.join(x.dropna().unique()[:10]))
    .reset_index()
)
descriptors.columns = ['Category', 'titles']
descriptors['text'] = 'Category: ' + descriptors['Category'] + '. Products: ' + descriptors['titles']

del data
gc.collect()

print(f'Descriptors built: {len(descriptors)} categories')
print(f'\nExample:\n{descriptors["text"].iloc[0]}')

Descriptors built: 1625 categories

Example:
Category: 3D_PRINTER. Products: ELEGOO 3D Printer Neptune 2S FDM 3D Printer with PEI Printing Sheet Large Printing Size 8.66x8.66x9.84 inch; Creality Ender 3 /Pro/V2 3D Printer Assembled Extruder MK8 HotEnd Kit 24V with 0.4mm Nozzle Upgrade with Low Friction Creality-Capricorn Tubing; Creality Original Ultra-Flexible Removable Magnetic 3D Printer Build Surface Heated Bed Cover for Ender 3 V2 Neo/Ender 3 pro/Ender 3 S1/Ender 5 3D Printer 235X235MM; Official Creality Ender 3 Pro 3D Printer with Resume Printing by MKK, Upgraded C-Magnet Build Surface Plate Mat, UL Certified Power Supply, Metal Frame FDM DIY Printers by MKK 220x220x250mm; ANTCLABS BLTouch : Auto Bed Leveling Sensor/to be a Premium 3D Printer (with 2M Extension Cable Set); Phrozen Sonic Mini LCD Resin 3D Printer Series, Matrix LED UV Light Tech, Monochrome/Mono LCD Screen, Longer Working Hours, for Jewelry-Making and Miniatures (Sonic Mini); Official Creality Ender 3 V2 3D Printe

In [4]:
import numpy as np
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2: ~80MB, 384-dim. Encoding 1,625 strings takes ~5-10s on CPU.
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(descriptors['text'].tolist(), show_progress_bar=True)
embeddings = normalize(embeddings)  # L2-normalize so KMeans behaves like cosine distance

np.save('../data/category_embeddings.npy', embeddings)
print(f'Embeddings shape: {embeddings.shape}')
print('Saved to data/category_embeddings.npy')

c:\Users\shaha\Documents\1. SU Masters\Code Space\group-project-bas-team\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 51/51 [00:12<00:00,  4.11it/s]

Embeddings shape: (1625, 384)
Saved to data/category_embeddings.npy


In [5]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

results = {}
for k in [15, 20, 25, 30]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels, sample_size=500, random_state=42)
    results[k] = (score, labels)
    print(f'k={k}  silhouette={score:.4f}')

best_k = max(results, key=lambda k: results[k][0])
descriptors['cluster_id'] = results[best_k][1]
print(f'\nSelected k={best_k}')

k=15  silhouette=0.0392
k=20  silhouette=0.0353
k=25  silhouette=0.0378
k=30  silhouette=0.0266

Selected k=15


In [6]:
# Print top categories per cluster so you can assign human-readable labels in the next cell.
# Load only the Category column to get row frequencies — cheap read.
data_temp = pd.read_csv('../data/data_unencoded.csv', usecols=['Category'])
freq = data_temp['Category'].value_counts().rename('row_count')
del data_temp; gc.collect()

cluster_view = (
    descriptors[['Category', 'cluster_id']]
    .merge(freq, left_on='Category', right_index=True)
    .sort_values(['cluster_id', 'row_count'], ascending=[True, False])
)

for cid, grp in cluster_view.groupby('cluster_id'):
    print(f'\n--- Cluster {cid} ({len(grp)} categories) ---')
    print(grp[['Category', 'row_count']].head(8).to_string(index=False))


--- Cluster 0 (125 categories) ---
              Category  row_count
          DRINKING_CUP        723
               KITCHEN        509
FOOD_STORAGE_CONTAINER        458
  POTABLE_WATER_FILTER        284
      FOOD_STORAGE_BAG        228
           THERMOMETER        195
            BAKING_PAN        191
               PLANTER        184

--- Cluster 1 (97 categories) ---
                        Category  row_count
                ELECTRONIC_CABLE       1725
             CELLULAR_PHONE_CASE       1333
                         BATTERY        969
                SCREEN_PROTECTOR        820
                CHARGING_ADAPTER        719
PORTABLE_ELECTRONIC_DEVICE_COVER        670
              ELECTRONIC_ADAPTER        432
                    FLASH_MEMORY        298

--- Cluster 2 (120 categories) ---
                Category  row_count
      INKJET_PRINTER_INK        383
         HVAC_AIR_FILTER        310
               AUTO_PART        218
         HARDWARE_TUBING        170
ELECTRONIC_

In [9]:
cluster_labels = {
    0:  'Kitchen_Home',
    1:  'Electronics_Accessories',
    2:  'Industrial_Auto',
    3:  'Home_Textiles',
    4:  'Consumer_Electronics',
    5:  'Toys_Games',
    6:  'Health_Beauty',
    7:  'Food_Nutrition',
    8:  'Books_Media',
    9:  'Home_Electrical',
    10: 'Office_Stationery',
    11: 'Sports_Fitness',
    12: 'Apparel',
    13: 'Tools_Hardware',
    14: 'Home_Decor',
}

descriptors['parent_label'] = descriptors['cluster_id'].map(cluster_labels)

mapping = descriptors[['Category', 'cluster_id', 'parent_label']].rename(
    columns={'cluster_id': 'parent_id'}
)
mapping.to_csv('../data/category_mapping.csv', index=False)
print(f'Mapping saved: {mapping.shape}')
print(mapping['parent_label'].value_counts())

Mapping saved: (1625, 3)
parent_label
Health_Beauty              154
Tools_Hardware             151
Office_Stationery          133
Kitchen_Home               125
Industrial_Auto            120
Home_Textiles              118
Food_Nutrition             106
Home_Decor                 104
Electronics_Accessories     97
Apparel                     94
Home_Electrical             87
Toys_Games                  87
Books_Media                 85
Consumer_Electronics        84
Sports_Fitness              80
Name: count, dtype: int64


In [10]:
def assign_price_tier(x):
    """Assign 0/1/2 (low/med/high) by tertile within a category. Falls back to mid-tier for tiny groups."""
    if len(x) < 3:
        return pd.Series([1] * len(x), index=x.index)
    try:
        return pd.qcut(x.rank(method='first'), q=3, labels=[0, 1, 2]).astype(int)
    except Exception:
        return pd.Series([1] * len(x), index=x.index)

encoded = pd.read_csv('../data/cleaned_data.csv')
unencoded = pd.read_csv('../data/data_unencoded.csv', usecols=['Category', 'Purchase Price Per Unit'])

unencoded['price_tier'] = (
    unencoded.groupby('Category')['Purchase Price Per Unit']
    .transform(assign_price_tier)
    .astype(int)
)

cat_to_parent = mapping.set_index('Category')['parent_id']
unencoded['parent_id'] = unencoded['Category'].map(cat_to_parent)

# copy() before adding columns avoids pandas fragmentation warning on the 156-col DataFrame
encoded = encoded.copy()
encoded['price_tier'] = unencoded['price_tier'].values
encoded['parent_category'] = unencoded['parent_id'].values

encoded.to_csv('../data/cleaned_data_with_parent.csv', index=False)
print(f'Saved cleaned_data_with_parent.csv: {encoded.shape}')
print(f'Unique parent categories: {encoded["parent_category"].nunique()}')
print(f'price_tier distribution:\n{encoded["price_tier"].value_counts().sort_index()}')

Saved cleaned_data_with_parent.csv: (157026, 158)
Unique parent categories: 15
price_tier distribution:
price_tier
0    52749
1    51993
2    52284
Name: count, dtype: int64
